# XGBoost Cross-Account Training with mlp_sdk

This notebook demonstrates how to train an XGBoost model on SageMaker **in a different AWS account**
using the mlp_sdk cross-account training wrapper.

## What You'll Learn

1. Generate synthetic training data
2. Initialize mlp_sdk session (source account)
3. Upload data to the target account's S3 bucket via assumed role
4. Train an XGBoost model in the target account using `run_training_job_xacct`
5. Monitor training progress in the target account
6. View audit trail

## Prerequisites

- mlp_sdk installed
- AWS credentials configured (source account)
- A cross-account IAM role in the target account with a trust policy allowing your source account
- A SageMaker execution role in the target account
- An S3 bucket in the target account for training data and output

## Two Roles Involved

- **target_role_arn**: The IAM role you assume via STS to access the target account
- **execution_role_arn**: The SageMaker execution role in the target account (passed as `role_arn`)

These can be the same role if it has both the trust policy and SageMaker permissions.

In [ ]:
!pip install -i https://test.pypi.org/simple/ mlp-sdk==0.1.1

## Step 1: Install Dependencies and Import Libraries

In [ ]:
#!pip install sagemaker-mlp-sdk numpy pandas scikit-learn boto3
!pip install numpy pandas scikit-learn boto3

In [ ]:
import numpy as np
import pandas as pd
import boto3
import os
import time
from datetime import datetime
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

from mlp_sdk import MLP_Session
from mlp_sdk.exceptions import MLPSDKError

print('All imports successful!')

## Step 2: Configure Cross-Account Parameters

Set the target account role ARNs and S3 bucket. Update these values for your environment.

In [ ]:
# ============================================================
# UPDATE THESE VALUES FOR YOUR ENVIRONMENT
# ============================================================

# IAM role in the target account to assume via STS
TARGET_ROLE_ARN = 'arn:aws:iam::624178040188:role/MLPXAcctTrainingRole'

# SageMaker execution role in the target account
EXECUTION_ROLE_ARN = 'arn:aws:iam::624178040188:role/MLPXAcctTrainingRole'

# S3 bucket in the target account
TARGET_BUCKET = 'sagemaker-mlp-us-west-2-624178040188'

# Optional: external ID for the trust policy (set to None if not required)
EXTERNAL_ID = 'mlp-xacct-training'

# Optional: target region (None = same as source account)
TARGET_REGION = 'us-west-2'

print(f'Target role:      {TARGET_ROLE_ARN}')
print(f'Execution role:   {EXECUTION_ROLE_ARN}')
print(f'Target bucket:    {TARGET_BUCKET}')
print(f'External ID:      {EXTERNAL_ID}')
print(f'Target region:    {TARGET_REGION or "(same as source)"}')

## Step 3: Generate Synthetic Training Data

We'll create a binary classification dataset with 10,000 samples and 20 features.

In [ ]:
np.random.seed(42)

X, y = make_classification(
    n_samples=10000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    weights=[0.7, 0.3],
    flip_y=0.05,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples:   {len(X_train)}')
print(f'Validation samples: {len(X_val)}')
print(f'Features:           {X_train.shape[1]}')
print(f'Class distribution: {np.bincount(y_train)}')

## Step 4: Prepare Data for XGBoost

XGBoost expects data in CSV format with no header and the target variable in the first column.

In [ ]:
train_df = pd.DataFrame(X_train)
train_df.insert(0, 'target', y_train)

val_df = pd.DataFrame(X_val)
val_df.insert(0, 'target', y_val)

os.makedirs('data', exist_ok=True)
train_df.to_csv('data/train.csv', header=False, index=False)
val_df.to_csv('data/validation.csv', header=False, index=False)

print(f'data/train.csv      ({os.path.getsize("data/train.csv")} bytes)')
print(f'data/validation.csv ({os.path.getsize("data/validation.csv")} bytes)')

## Step 5: Initialize mlp_sdk Session

This initializes the session in your source account. The cross-account role assumption happens later.

In [ ]:
try:
    # Initialize session with default config
    # If you have a custom config path, use: MLP_Session(config_path="/path/to/config.yaml")
    session = MLP_Session(config_path="admin-config-xacct.yaml",log_level="DEBUG")
    
    print("✅ MLP_Session initialized successfully!")
    print(f"   Region: {session.region_name}")
    print(f"   Default bucket: {session.default_bucket}")
    print(f"   Execution role: {session.get_execution_role()}")
    
    # View configuration
    config = session.config_manager.MLP_config
    print(f"\n📋 Configuration loaded:")
    print(f"   Training instance: {config.compute_config.training_instance_type}")
    print(f"   Instance count: {config.compute_config.training_instance_count}")
    print(f"   VPC: {config.networking_config.vpc_id}")
    
except Exception as e:
    print(f"❌ Error initializing session: {e}")
    print("\n💡 Tip: Generate config with: python examples/generate_admin_config.py --interactive")
    raise

## Step 6: Upload Data to Target Account S3

Assume the target role to upload training data to the target account's S3 bucket.

In [ ]:
# Assume role to access target account S3
sts_client = boto3.client('sts')
assume_params = {
    'RoleArn': TARGET_ROLE_ARN,
    'RoleSessionName': 'mlp-xacct-upload',
    'DurationSeconds': 3600,
}
if EXTERNAL_ID:
    assume_params['ExternalId'] = EXTERNAL_ID

print(f'🔑 Assuming role: {TARGET_ROLE_ARN}')
sts_response = sts_client.assume_role(**assume_params)
creds = sts_response['Credentials']
print(f'✅ Role assumed successfully')

# Create S3 client with assumed credentials
xacct_session = boto3.Session(
    aws_access_key_id=creds['AccessKeyId'],
    aws_secret_access_key=creds['SecretAccessKey'],
    aws_session_token=creds['SessionToken'],
    region_name=TARGET_REGION or boto3.Session().region_name,
)
s3_client = xacct_session.client('s3')

# Upload data
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
s3_prefix = f'xgboost-xacct-example/{timestamp}'

train_s3_path = f's3://{TARGET_BUCKET}/{s3_prefix}/train/'
val_s3_path = f's3://{TARGET_BUCKET}/{s3_prefix}/validation/'
output_s3_path = f's3://{TARGET_BUCKET}/{s3_prefix}/output'

print(f'\n📤 Uploading to: s3://{TARGET_BUCKET}/{s3_prefix}')

s3_client.upload_file('data/train.csv', TARGET_BUCKET, f'{s3_prefix}/train/train.csv')
print(f'   ✅ {train_s3_path}')

s3_client.upload_file('data/validation.csv', TARGET_BUCKET, f'{s3_prefix}/validation/validation.csv')
print(f'   ✅ {val_s3_path}')

## Step 7: Configure XGBoost Training Job

Define the training job parameters. The mlp_sdk will automatically apply defaults from your configuration.

In [ ]:
hyperparameters = {
    'objective': 'binary:logistic',
    'num_round': '100',
    'max_depth': '5',
    'eta': '0.2',
    'gamma': '4',
    'min_child_weight': '6',
    'subsample': '0.8',
    'verbosity': '1',
    'eval_metric': 'auc',
    'scale_pos_weight': '2'
}

region = TARGET_REGION or session.region_name
xgboost_container = f'246618743249.dkr.ecr.{region}.amazonaws.com/sagemaker-xgboost:1.5-1'

print('📋 Training configuration:')
print(f'   Container:        {xgboost_container}')
print(f'   Hyperparameters:  {len(hyperparameters)} parameters')
print(f'   Training data:    {train_s3_path}')
print(f'   Validation data:  {val_s3_path}')
print(f'   Output path:      {output_s3_path}')
print(f'   Target role:      {TARGET_ROLE_ARN}')
print(f'   Execution role:   {EXECUTION_ROLE_ARN}')

## Step 8: Start Cross-Account Training Job with mlp_sdk

Use `run_training_job_xacct` to assume the target role and run the training job in the target account.

This method:
1. Calls STS AssumeRole with `target_role_arn`
2. Creates a new SageMaker session with temporary credentials
3. Delegates to `run_training_job` using that cross-account session

Config defaults (instance type, VPC, KMS, etc.) are still applied from your local admin-config.yaml.

In [ ]:
job_name = f'xgboost-xacct-{timestamp}'

print(f'🚀 Starting cross-account training job: {job_name}')
print(f'\n⏳ This may take 5-10 minutes...\n')

try:
    inputs = {
        'train': train_s3_path,
        'validation': val_s3_path
    }

    xacct_kwargs = {
        'hyperparameters': hyperparameters,
        'output_path': output_s3_path,
        'max_run_in_seconds': 3600,
        'role_arn': EXECUTION_ROLE_ARN,
    }
    if TARGET_REGION:
        xacct_kwargs['target_region'] = TARGET_REGION

    training_job = session.run_training_job_xacct(
        job_name=job_name,
        target_role_arn=TARGET_ROLE_ARN,
        training_image=xgboost_container,
        inputs=inputs,
        external_id=EXTERNAL_ID,
        **xacct_kwargs
    )

    print(f'\n✅ Cross-account training job started!')
    print(f'   ModelTrainer object created')
    print(f'\n💡 Monitor in the TARGET account SageMaker console')

except MLPSDKError as e:
    print(f'❌ SDK Error: {e}')
    raise
except Exception as e:
    print(f'❌ Error: {e}')
    raise

## Step 9: Monitor Training Progress

Monitor the training job status in the target account by assuming the role again.

In [ ]:
# Get actual job name from ModelTrainer
if hasattr(training_job, '_latest_training_job') and training_job._latest_training_job:
    actual_job_name = training_job._latest_training_job.get_name()
    job_name = actual_job_name
    print(f'✅ Actual training job name: {job_name}')

# Assume role to poll status in target account
assume_params = {
    'RoleArn': TARGET_ROLE_ARN,
    'RoleSessionName': 'mlp-xacct-monitor',
    'DurationSeconds': 3600,
}
if EXTERNAL_ID:
    assume_params['ExternalId'] = EXTERNAL_ID

creds = boto3.client('sts').assume_role(**assume_params)['Credentials']
sm_client = boto3.Session(
    aws_access_key_id=creds['AccessKeyId'],
    aws_secret_access_key=creds['SecretAccessKey'],
    aws_session_token=creds['SessionToken'],
    region_name=TARGET_REGION or boto3.Session().region_name,
).client('sagemaker')

print(f'📊 Monitoring training job: {job_name}\n')

while True:
    response = sm_client.describe_training_job(TrainingJobName=job_name)
    status = response['TrainingJobStatus']

    if status == 'Completed':
        print(f'\n✅ Training completed successfully!')
        print(f'   Training time: {response.get("TrainingTimeInSeconds", 0)} seconds')
        print(f'   Billable time: {response.get("BillableTimeInSeconds", 0)} seconds')

        if 'FinalMetricDataList' in response:
            print(f'\n📈 Final metrics:')
            for metric in response['FinalMetricDataList']:
                print(f'   {metric["MetricName"]}: {metric["Value"]:.4f}')

        model_artifacts = response['ModelArtifacts']['S3ModelArtifacts']
        print(f'\n📦 Model artifacts: {model_artifacts}')
        break

    elif status == 'Failed':
        print(f'\n❌ Training failed!')
        print(f'   Failure reason: {response.get("FailureReason", "Unknown")}')
        break

    elif status == 'Stopped':
        print(f'\n⚠️  Training stopped!')
        break

    else:
        print(f'   Status: {status} | Time: {datetime.now().strftime("%H:%M:%S")}', end='\r')
        time.sleep(30)

## Step 10: View Training Logs (Optional)

Training logs are in the **target account's** CloudWatch.

In [ ]:
print('📝 Training logs (in TARGET account):')
print(f'   Log group: /aws/sagemaker/TrainingJobs')
print(f'   Log stream: {job_name}/algo-1-*')
print(f'\n💡 View logs in the TARGET account CloudWatch console or use AWS CLI:')
print(f'   aws logs tail /aws/sagemaker/TrainingJobs --follow --log-stream-name-prefix {job_name}')

## Step 11: View Audit Trail

The mlp_sdk automatically tracks cross-account operations in the audit trail.

In [ ]:
audit_entries = session.get_audit_trail(operation='run_training_job_xacct')

print(f'📊 Audit Trail ({len(audit_entries)} cross-account training operations):\n')

for entry in audit_entries[-5:]:
    print(f'   {entry.get("timestamp")}: {entry.get("operation")}')
    print(f'      Status: {entry.get("status")}')
    if 'parameters' in entry:
        print(f'      Job: {entry["parameters"].get("job_name", "N/A")}')
    print()

## Summary

This notebook demonstrated cross-account SageMaker training using `mlp_sdk.run_training_job_xacct()`.

Key points:
- `target_role_arn` is assumed via STS to access the target account
- `role_arn` (execution role) is used by SageMaker to run the training job
- Config defaults (instance type, VPC, KMS) are still applied from your local config
- Monitoring requires assuming the role again since the job runs in the target account
- All operations are tracked in the audit trail